# Plate 04 · Cell Recogniser Training (CRNN-CTC on device, PARSeq on server)

Kuwaiti plates contain **only digits**, so the recogniser reads short **cell strips** (32 px high, shape kept, padded to 160 px):
the code and the number, each in Arabic-Indic (top row) and English (bottom row) digits.
Vocabulary: 10 Arabic-Indic + 10 English digits (+ CTC blank) - much easier than letters.

**Dual-row cross-check:** the app reads both rows of each cell and compares them
(`Plate_Normalizer.cross_check`). Agreement → confident result; disagreement → server fallback / worker confirms.
This is the biggest accuracy gain specific to Kuwaiti plates.

**On-device recogniser – CRNN:** MobileNetV3-Small feature extractor → BiLSTM (2×96) → CTC. ≈1 M params,
**≈3 MB int8**, ~2 ms per cell. **Server fallback – PARSeq** (Apache-2.0) when the phone's confidence < 0.85.

Data: real cells (notebook 01) + synthetic cells (notebook 02), 1:1 at the start, real-only for the last epochs.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision
from torchvision import transforms
from Common_Utils.Plate_Normalizer import CHARSET
from Common_Utils.Kuwait_Plate_Layout import CELL_W, CELL_H

BLANK = 0
VOCAB = ["<blank>"] + list(CHARSET)          # 10 Arabic-Indic + 10 English digits
CHAR_TO_ID = {c: i for i, c in enumerate(VOCAB)}
device = "cuda" if torch.cuda.is_available() else "cpu"
len(VOCAB)

In [ ]:
class Plate_Cells(Dataset):
    def __init__(self, root, labels_csv, train=True):
        self.root, self.df = Path(root), pd.read_csv(labels_csv)
        aug = [transforms.RandomAffine(2, translate=(0.03, 0.06), shear=3), transforms.ColorJitter(0.4, 0.4)] if train else []
        self.tf = transforms.Compose(aug + [transforms.Grayscale(), transforms.Resize((CELL_H, CELL_W)), transforms.ToTensor(),
                                            transforms.Normalize([0.5], [0.5])])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = self.tf(Image.open(self.root / row["file"]).convert("RGB"))
        target = torch.tensor([CHAR_TO_ID[c] for c in str(row["text"]) if c in CHAR_TO_ID], dtype=torch.long)
        return image, target

def collate(batch):
    images, targets = zip(*batch)
    lengths = torch.tensor([len(t) for t in targets])
    return torch.stack(images), torch.cat(targets), lengths

In [ ]:
class CRNN(nn.Module):
    """MobileNetV3-Small backbone + BiLSTM + CTC head, for 32 x 160 digit cells (20 time steps)."""
    def __init__(self, num_classes=len(VOCAB)):
        super().__init__()
        mb = torchvision.models.mobilenet_v3_small(weights="DEFAULT").features
        mb[0][0] = nn.Conv2d(1, 16, 3, stride=(2, 1), padding=1, bias=False)   # grayscale, keep width
        self.backbone = mb[:9]                                              # -> C=48
        self.rnn = nn.LSTM(48, 96, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1)
        self.head = nn.Linear(192, num_classes)
    def forward(self, x):                       # x: B x 1 x 32 x 160
        f = self.backbone(x).mean(dim=2).permute(0, 2, 1)   # average over height (ONNX/TFLite friendly) -> B x T x C
        out, _ = self.rnn(f)
        return self.head(out).log_softmax(-1)  # B x T x K

model = CRNN().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")
print("time steps:", model(torch.zeros(1, 1, CELL_H, CELL_W, device=device)).shape)   # needs T >= 2*5+1 for 5 digits

In [ ]:
def greedy_decode(log_probs):
    best = log_probs.argmax(-1).cpu().tolist()
    texts, confs = [], []
    probs = log_probs.exp().max(-1).values.cpu()
    for seq, p in zip(best, probs):
        chars, prev = [], BLANK
        for t in seq:
            if t != prev and t != BLANK:
                chars.append(VOCAB[t])
            prev = t
        texts.append("".join(chars))
        confs.append(float(p.min()))            # min-prob: one doubtful digit makes the whole cell doubtful
    return texts, confs

REC = DATA / "Plates" / "Recognition"
SYN = DATA / "Plates" / "Synthetic"
parts = [Plate_Cells(p, p / "labels.csv") for p in (REC, SYN) if (p / "labels.csv").exists()]
train_ds = ConcatDataset(parts)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, collate_fn=collate)
len(train_ds)

In [ ]:
from Common_Utils.Metrics_Utils import exact_match, character_error_rate

ctc = nn.CTCLoss(blank=BLANK, zero_infinity=True)
optim = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
EPOCHS = 60
sched = torch.optim.lr_scheduler.OneCycleLR(optim, max_lr=3e-3, total_steps=EPOCHS * max(1, len(train_dl)))
scaler = torch.amp.GradScaler(enabled=device == "cuda")

for epoch in range(EPOCHS):
    model.train()
    for images, targets, lengths in train_dl:
        images = images.to(device)
        with torch.autocast(device, enabled=device == "cuda"):
            log_probs = model(images)
        T = log_probs.size(1)
        loss = ctc(log_probs.float().permute(1, 0, 2), targets, torch.full((images.size(0),), T, dtype=torch.long), lengths)
        optim.zero_grad(); scaler.scale(loss).backward(); scaler.step(optim); scaler.update(); sched.step()
    print(f"epoch {epoch:03d} loss {loss.item():.4f}")
    torch.save(model.state_dict(), RUNS / "crnn_last.pt")

### Server model: PARSeq fine-tune
```bash
git clone https://github.com/baudm/parseq && cd parseq && pip install -r requirements/core.txt -e .
# Build an LMDB from Data/Plates/Recognition + Synthetic/cells with tools/create_lmdb_dataset.py
./train.py pretrained=parseq-tiny charset_train="٠١٢٣٤٥٦٧٨٩0123456789" \
           data.root_dir=<lmdb_dir> trainer.max_epochs=30 model.lr=2e-4
```
Export the best checkpoint to ONNX (notebook 06) for the FastAPI inference service.